In [ ]:
# Install the required packages in Google Colab
!pip install -q --upgrade yfinance openpyxl

# Import libraries
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import yfinance as yf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.1/144.1 kB 1.9 MB/s eta 0:00:00


In [ ]:
# User inputs
ticker = input("Enter the Yahoo Finance ticker, for example AAPL: ").strip().upper()
start_date = input("Enter start date in YYYY-MM-DD format: ").strip()
end_date = input("Enter end date in YYYY-MM-DD format: ").strip()

# Example:
# ticker = "AAPL"
# start_date = "2020-01-01"
# end_date = "2026-07-16"

Enter the Yahoo Finance ticker, for example AAPL: AAPL
Enter start date in YYYY-MM-DD format: 2026-07-16
Enter end date in YYYY-MM-DD format: 2026-08-11


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def download_stock_dataset(
    ticker: str,
    start_date: str,
    end_date: str
) -> tuple[pd.DataFrame, dict]:
    """
    Download historical stock data from Yahoo Finance and prepare
    a research-ready dataset.

    Parameters
    ----------
    ticker : str
        Yahoo Finance ticker symbol, such as AAPL.
    start_date : str
        Beginning date in YYYY-MM-DD format.
    end_date : str
        Ending date in YYYY-MM-DD format.

    Returns
    -------
    stock_data : pandas.DataFrame
        Historical stock-price dataset with calculated variables.
    company_info : dict
        Selected company metadata.
    """

    # Validate input
    if not ticker:
        raise ValueError("A ticker symbol is required.")

    try:
        start = pd.to_datetime(start_date)
        end = pd.to_datetime(end_date)
    except ValueError as error:
        raise ValueError(
            "Dates must use the YYYY-MM-DD format."
        ) from error

    if start >= end:
        raise ValueError("The start date must be earlier than the end date.")

    # Create Yahoo Finance ticker object
    company = yf.Ticker(ticker)

    # Download historical daily stock data
    stock_data = yf.download(
        tickers=ticker,
        start=start_date,
        end=end_date,
        interval="1d",
        auto_adjust=False,
        actions=True,
        progress=False,
        threads=False
    )

    if stock_data.empty:
        raise ValueError(
            f"No data were returned for '{ticker}'. "
            "Check the ticker and date range."
        )

    # Flatten column names if yfinance returns a MultiIndex
    if isinstance(stock_data.columns, pd.MultiIndex):
        stock_data.columns = stock_data.columns.get_level_values(0)

    # Move the trading date from the index into a regular column
    stock_data = stock_data.reset_index()

    # Standardize column names
    stock_data.columns = [
        str(column).strip().replace(" ", "_")
        for column in stock_data.columns
    ]

    # Add company identifier
    stock_data.insert(1, "Ticker", ticker)

    # Calculate daily price change
    stock_data["Daily_Price_Change"] = stock_data["Close"].diff()

    # Calculate simple daily return
    stock_data["Daily_Return"] = stock_data["Close"].pct_change()

    # Calculate log return
    stock_data["Log_Return"] = np.log(
        stock_data["Close"] / stock_data["Close"].shift(1)
    )

    # Calculate intraday price range
    stock_data["Daily_Range"] = (
        stock_data["High"] - stock_data["Low"]
    )

    # Calculate intraday percentage range
    stock_data["Daily_Range_Percent"] = (
        stock_data["Daily_Range"] / stock_data["Open"]
    ) * 100

    # Calculate rolling averages
    stock_data["Moving_Average_7_Day"] = (
        stock_data["Close"].rolling(window=7).mean()
    )

    stock_data["Moving_Average_30_Day"] = (
        stock_data["Close"].rolling(window=30).mean()
    )

    # Calculate 30-trading-day volatility
    stock_data["Volatility_30_Day"] = (
        stock_data["Daily_Return"]
        .rolling(window=30)
        .std()
    )

    # Calculate annualized rolling volatility
    stock_data["Annualized_Volatility_30_Day"] = (
        stock_data["Volatility_30_Day"] * np.sqrt(252)
    )

    # Calculate trading-volume change
    stock_data["Volume_Change_Percent"] = (
        stock_data["Volume"].pct_change() * 100
    )

    # Try to collect selected company metadata
    try:
        info = company.info

        company_info = {
            "Ticker": ticker,
            "Company_Name": info.get("longName"),
            "Exchange": info.get("exchange"),
            "Currency": info.get("currency"),
            "Sector": info.get("sector"),
            "Industry": info.get("industry"),
            "Country": info.get("country"),
            "Website": info.get("website")
        }

    except Exception:
        # Metadata availability can vary by ticker
        company_info = {
            "Ticker": ticker,
            "Company_Name": None,
            "Exchange": None,
            "Currency": None,
            "Sector": None,
            "Industry": None,
            "Country": None,
            "Website": None
        }

    return stock_data, company_info

'''
try:
    stock_df, company_metadata = download_stock_dataset(
        ticker=ticker,
        start_date=start_date,
        end_date=end_date
    )

    print(f"Successfully downloaded {len(stock_df):,} observations.")
    print(f"Date range: {stock_df['Date'].min()} through {stock_df['Date'].max()}")

except Exception as error:
    print(f"Unable to create dataset: {error}")
    raise
'''

'\ntry:\n    stock_df, company_metadata = download_stock_dataset(\n        ticker=ticker,\n        start_date=start_date,\n        end_date=end_date\n    )\n\n    print(f"Successfully downloaded {len(stock_df):,} observations.")\n    print(f"Date range: {stock_df[\'Date\'].min()} through {stock_df[\'Date\'].max()}")\n\nexcept Exception as error:\n    print(f"Unable to create dataset: {error}")\n    raise\n'

In [ ]:
try:
    stock_df, company_metadata = download_stock_dataset(
        ticker=ticker,
        start_date=start_date,
        end_date=end_date
    )

    print(f"Successfully downloaded {len(stock_df):,} observations.")
    print(f"Date range: {stock_df['Date'].min()} through {stock_df['Date'].max()}")

except Exception as error:
    print(f"Unable to create dataset: {error}")
    raise

Successfully downloaded 18 observations.
Date range: 2026-07-16 00:00:00 through 2026-08-10 00:00:00


In [ ]:
# Display company information
company_metadata_df = pd.DataFrame([company_metadata])

print("Company information:")
display(company_metadata_df)

print("\nFirst five stock observations:")
display(stock_df.head())

print("\nDataset structure:")
stock_df.info()

print("\nDescriptive statistics:")
display(stock_df.describe(include="all").transpose())

Company information:


,Ticker,Company_Name,Exchange,Currency,Sector,Industry,Country,Website
0,AAPL,Apple Inc.,NMS,USD,Technology,Consumer Electronics,United States,https://www.apple.com



First five stock observations:


,Date,Ticker,Adj_Close,Close,Dividends,High,Low,Open,Stock_Splits,Volume,Daily_Price_Change,Daily_Return,Log_Return,Daily_Range,Daily_Range_Percent,Moving_Average_7_Day,Moving_Average_30_Day,Volatility_30_Day,Annualized_Volatility_30_Day,Volume_Change_Percent
0,2026-07-16,AAPL,332.972839,333.260010,0.0,334.679993,326.790009,328.010010,0.0,62970600,NaN,NaN,NaN,7.889984,2.405410,NaN,NaN,NaN,NaN,NaN
1,2026-07-17,AAPL,333.452393,333.739990,0.0,334.989990,329.000000,331.980011,0.0,63407100,0.479980,0.001440,0.001439,5.989990,1.804323,NaN,NaN,NaN,NaN,0.693181
2,2026-07-20,AAPL,326.308563,326.589996,0.0,333.709991,323.679993,333.510010,0.0,53468000,-7.149994,-0.021424,-0.021657,10.029999,3.007406,NaN,NaN,NaN,NaN,-15.675058
3,2026-07-21,AAPL,327.457581,327.739990,0.0,329.600006,322.220001,323.130005,0.0,41338900,1.149994,0.003521,0.003515,7.380005,2.283912,NaN,NaN,NaN,NaN,-22.684783
4,2026-07-22,AAPL,325.609192,325.890015,0.0,329.000000,323.339996,327.869995,0.0,38755900,-1.849976,-0.005645,-0.005661,5.660004,1.726295,NaN,NaN,NaN,NaN,-6.248352



Dataset structure:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 20 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Date                          18 non-null     datetime64[ns]
 1   Ticker                        18 non-null     object        
 2   Adj_Close                     18 non-null     float64       
 3   Close                         18 non-null     float64       
 4   Dividends                     18 non-null     float64       
 5   High                          18 non-null     float64       
 6   Low                           18 non-null     float64       
 7   Open                          18 non-null     float64       
 8   Stock_Splits                  18 non-null     float64       
 9   Volume                        18 non-null     int64         
 10  Daily_Price_Change            17 non-null     float64       
 11  Daily_Return  

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Date,18,NaN,NaN,NaN,2026-07-28 06:40:00,2026-07-16 00:00:00,2026-07-22 06:00:00,2026-07-28 12:00:00,2026-08-03 18:00:00,2026-08-10 00:00:00,NaN
Ticker,18,1,AAPL,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Adj_Close,18.0,NaN,NaN,NaN,322.915165,303.158569,311.084221,325.958878,333.100212,339.786926,12.157802
Close,18.0,NaN,NaN,NaN,323.178889,303.420013,311.352501,326.240005,333.387497,340.079987,12.187289
Dividends,18.0,NaN,NaN,NaN,0.015,0.0,0.0,0.0,0.0,0.27,0.06364
High,18.0,NaN,NaN,NaN,325.856111,308.26001,312.55249,329.300003,334.732498,344.570007,12.420452
Low,18.0,NaN,NaN,NaN,318.705,300.0,306.560013,321.919998,328.447502,337.350006,12.507737
Open,18.0,NaN,NaN,NaN,321.917781,302.730011,310.047493,322.460007,332.820007,340.029999,12.378323
Stock_Splits,18.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Volume,18.0,NaN,NaN,NaN,57278505.555556,34437200.0,45144350.0,50731650.0,63297975.0,132489100.0,22222303.02555


In [ ]:
# Create a directory for the output
output_directory = Path("/content/drive/MyDrive/ColabNotebooks/DataForReputation")
output_directory.mkdir(parents=True, exist_ok=True)

# Create safe file names
safe_start_date = start_date.replace("-", "")
safe_end_date = end_date.replace("-", "")

csv_filename = (
    f"{ticker}_stock_data_{safe_start_date}_{safe_end_date}.csv"
)

excel_filename = (
    f"{ticker}_stock_data_{safe_start_date}_{safe_end_date}.xlsx"
)

metadata_filename = (
    f"{ticker}_company_metadata.csv"
)

csv_path = output_directory / csv_filename
excel_path = output_directory / excel_filename
metadata_path = output_directory / metadata_filename

# Save stock data as CSV
stock_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

# Save company metadata as CSV
company_metadata_df.to_csv(
    metadata_path,
    index=False,
    encoding="utf-8-sig"
)

# Save data and metadata in one Excel workbook
with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:
    stock_df.to_excel(
        writer,
        sheet_name="Stock Data",
        index=False
    )

    company_metadata_df.to_excel(
        writer,
        sheet_name="Company Information",
        index=False
    )

print("Files saved successfully:")
print(f"CSV dataset: {csv_path}")
print(f"Excel workbook: {excel_path}")
print(f"Metadata file: {metadata_path}")

Files saved successfully:
CSV dataset: /content/drive/My Drive/ColabNotebooks/DataForReputation/AAPL_stock_data_20260716_20260811.csv
Excel workbook: /content/drive/My Drive/ColabNotebooks/DataForReputation/AAPL_stock_data_20260716_20260811.xlsx
Metadata file: /content/drive/My Drive/ColabNotebooks/DataForReputation/AAPL_company_metadata.csv


In [ ]:
# Display files in the output directory
saved_files = pd.DataFrame({
    "File_Name": [file.name for file in output_directory.iterdir()],
    "File_Path": [str(file) for file in output_directory.iterdir()],
    "Size_KB": [
        round(file.stat().st_size / 1024, 2)
        for file in output_directory.iterdir()
    ]
})

display(saved_files)

,File_Name,File_Path,Size_KB
0,Code,/content/drive/My Drive/ColabNotebooks/DataFor...,4.00
1,AAPL_stock_data_20200101_20260719.csv,/content/drive/My Drive/ColabNotebooks/DataFor...,505.20
2,AAPL_company_metadata.csv,/content/drive/My Drive/ColabNotebooks/DataFor...,0.16
3,AAPL_stock_data_20200101_20260719.xlsx,/content/drive/My Drive/ColabNotebooks/DataFor...,301.04
4,AAPL_Apple_Inc_2026-07-16_to_2026-08-11_gdelt_...,/content/drive/My Drive/ColabNotebooks/DataFor...,318.27
5,AAPL_Apple_Inc_2026-07-16_to_2026-08-11_gdelt_...,/content/drive/My Drive/ColabNotebooks/DataFor...,162.04
6,AAPL_2026-07-16_to_2026-08-11_gdelt_request_lo...,/content/drive/My Drive/ColabNotebooks/DataFor...,0.24
7,AAPL_stock_data_20260716_20260811.csv,/content/drive/My Drive/ColabNotebooks/DataFor...,4.58
8,AAPL_stock_data_20260716_20260811.xlsx,/content/drive/My Drive/ColabNotebooks/DataFor...,8.70


In [ ]:
from google.colab import files

# Download the CSV dataset
files.download(str(csv_path))

# To download the Excel workbook instead, use:
files.download(str(excel_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>